[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter8/simple-table-chunking.ipynb)

# Chapter 8: Table Extraction and Chunking

Tables pose a unique challenge for RAG chunking: naive text splitting breaks rows across chunks, destroying the column-header relationships that give tabular data its meaning. A single product row split from its header becomes meaningless. This notebook demonstrates the problem by showing what happens when standard text splitters are applied to Markdown tables.

This notebook shows table-to-text conversion strategies and illustrates chunking pitfalls with tabular data.

**What you'll learn:**
- Convert DataFrames with multi-level columns to Markdown and JSON formats
- Observe how standard text splitting destroys table structure
- Understand why tables need specialized chunking strategies

**Prerequisites:** `pip install langchain-text-splitters pandas tabulate`

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
import json

## Create Sample Table

We build a product pricing DataFrame with multi-level column headers to simulate the kind of structured tables commonly found in business documents.

In [2]:
data_rows = [
    ['Model-A', '$299', '€257.25', '4.5 Stars'],
    ['Model-B', '$449', '€386.95', '4.2 Stars'],
    ['Model-C', '$199', '€171.50', '3.9 Stars'],
    ['Model-X', '$899', '€774.77', '4.8 Stars'],
    ['Model-Pro', '$1299', '€1122.15', '4.9 Stars'],
    ['Model-Lite', '$149', '€128.70', '3.5 Stars'],
    ['Model-Air', '$699', '€603.75', '4.4 Stars'],
    ['Model-Eco', '$99', '€85.52', '3.1 Stars'],
    ['Model-Max', '$1099', '€949.20', '4.7 Stars'],
    ['Model-Std', '$249', '€215.04', '4.0 Stars'],
    ['Model-Ultra', '$1599', '€1381.13', '4.8 Stars'],
    ['Model-Mini', '$349', '€301.44', '4.1 Stars'],
    ['Model-Go', '$179', '€154.60', '3.7 Stars'],
    ['Model-Titan', '$1999', '€1726.63', '4.9 Stars']
]
    
# Define the multi-level column structure as a list of tuples.
# Format: (Top-Level-Header, Sub-Level-Header)
column_structure = [
    ('Product', ''),  # Use an empty string for columns with no sub-header
    ('Price', 'USD'),
    ('Price', 'EUR'),
    ('Rating', '')
]

# --- 2. Create the DataFrame ---

# Create the MultiIndex object from our tuples
multi_cols = pd.MultiIndex.from_tuples(column_structure)

# Create the DataFrame
df = pd.DataFrame(data_rows, columns=multi_cols)

In [3]:
df

Product  Price               Rating
                   USD       EUR           
0       Model-A   $299   €257.25  4.5 Stars
1       Model-B   $449   €386.95  4.2 Stars
2       Model-C   $199   €171.50  3.9 Stars
3       Model-X   $899   €774.77  4.8 Stars
4     Model-Pro  $1299  €1122.15  4.9 Stars
5    Model-Lite   $149   €128.70  3.5 Stars
6     Model-Air   $699   €603.75  4.4 Stars
7     Model-Eco    $99    €85.52  3.1 Stars
8     Model-Max  $1099   €949.20  4.7 Stars
9     Model-Std   $249   €215.04  4.0 Stars
10  Model-Ultra  $1599  €1381.13  4.8 Stars
11   Model-Mini   $349   €301.44  4.1 Stars
12     Model-Go   $179   €154.60  3.7 Stars
13  Model-Titan  $1999  €1726.63  4.9 Stars

## Convert Table Formats

We export the table to JSON and Markdown formats to see how each serialization represents multi-level column headers before chunking.

In [4]:
parsed = json.loads(df.head(4).to_json())
print(json.dumps(parsed, indent=4))

{
    "('Product', '')": {
        "0": "Model-A",
        "1": "Model-B",
        "2": "Model-C",
        "3": "Model-X"
    },
    "('Price', 'USD')": {
        "0": "$299",
        "1": "$449",
        "2": "$199",
        "3": "$899"
    },
    "('Price', 'EUR')": {
        "0": "\u20ac257.25",
        "1": "\u20ac386.95",
        "2": "\u20ac171.50",
        "3": "\u20ac774.77"
    },
    "('Rating', '')": {
        "0": "4.5 Stars",
        "1": "4.2 Stars",
        "2": "3.9 Stars",
        "3": "4.8 Stars"
    }
}


In [5]:
raw_text = df.to_markdown()
raw_text

"|    | ('Product', '')   | ('Price', 'USD')   | ('Price', 'EUR')   | ('Rating', '')   |\n|---:|:------------------|:-------------------|:-------------------|:-----------------|\n|  0 | Model-A           | $299               | €257.25            | 4.5 Stars        |\n|  1 | Model-B           | $449               | €386.95            | 4.2 Stars        |\n|  2 | Model-C           | $199               | €171.50            | 3.9 Stars        |\n|  3 | Model-X           | $899               | €774.77            | 4.8 Stars        |\n|  4 | Model-Pro         | $1299              | €1122.15           | 4.9 Stars        |\n|  5 | Model-Lite        | $149               | €128.70            | 3.5 Stars        |\n|  6 | Model-Air         | $699               | €603.75            | 4.4 Stars        |\n|  7 | Model-Eco         | $99                | €85.52             | 3.1 Stars        |\n|  8 | Model-Max         | $1099              | €949.20            | 4.7 Stars        |\n|  9 | Model-Std    

## Demonstrate Chunking

Using a deliberately small chunk size, we show how standard text splitting tears apart table rows from their headers, destroying the structured relationships that give the data meaning.

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,  # Intentionally small to show the break
    chunk_overlap=20   # A small overlap
)

# Let's create the chunks
chunks = text_splitter.split_text(raw_text)

print(f"\n--- SPLIT INTO {len(chunks)} CHUNKS (Chunk Size: 100) ---")

# Now, let's see what's in each chunk
for i, chunk in enumerate(chunks):
    print(f"\n[CHUNK {i+1}]")
    # We use repr() to clearly show the newline characters (\n)
    print(repr(chunk))


--- SPLIT INTO 16 CHUNKS (Chunk Size: 100) ---

[CHUNK 1]
"|    | ('Product', '')   | ('Price', 'USD')   | ('Price', 'EUR')   | ('Rating', '')   |"

[CHUNK 2]
'|---:|:------------------|:-------------------|:-------------------|:-----------------|'

[CHUNK 3]
'|  0 | Model-A           | $299               | €257.25            | 4.5 Stars        |'

[CHUNK 4]
'|  1 | Model-B           | $449               | €386.95            | 4.2 Stars        |'

[CHUNK 5]
'|  2 | Model-C           | $199               | €171.50            | 3.9 Stars        |'

[CHUNK 6]
'|  3 | Model-X           | $899               | €774.77            | 4.8 Stars        |'

[CHUNK 7]
'|  4 | Model-Pro         | $1299              | €1122.15           | 4.9 Stars        |'

[CHUNK 8]
'|  5 | Model-Lite        | $149               | €128.70            | 3.5 Stars        |'

[CHUNK 9]
'|  6 | Model-Air         | $699               | €603.75            | 4.4 Stars        |'

[CHUNK 10]
'|  7 | Model-Eco         | $99